In [1]:
from gorgo import infer, condition, draw_from, flip, keep_deterministic, mem, factor
from gorgo.hashable import hashabledict
from gorgo.distributions.builtin_dists import Uniform, Beta
from gorgo.distributions import Mixture

from model import Utterance, Instance, Kind, meaning, literal_listener, speaker, pragmatic_listener

import numpy as np
from gorgo.inference import MaximumMarginalAPosteriori
import pickle

import pandas as pd

# Define helper functions

In [ ]:
##### Convert gorgo dist to df for export
@keep_deterministic
def dist_to_df(dist) -> pd.DataFrame:
    
    # validate input
    if not hasattr(dist, 'support') or not hasattr(dist, 'probabilities'):
        raise AttributeError("Distribution must have support and probabilities")
    
    # create dataframe based on support type
    if isinstance(dist.support[0], float): 
        df_dist = pd.DataFrame({
            'Element': dist.support,
            'Probability': dist.probabilities
        })
    else: 
        # for complex support, expand into multiple columns
        df_dist = pd.DataFrame(dist.support)
        df_dist['Probability'] = dist.probabilities
    
    return df_dist

# Set up Study 6 structure

In [3]:

###########################
## Set up Study 6 structure
###########################

# conditions
conditions = ("generic", "baseline", "specific")

# training trials per condition
generic_condition = (
    (Utterance("Zarpies", "love to eat flowers"), Instance("Zarpie", ("love to eat flowers", ))),
    (Utterance("Zarpies", "have stripes in their hair"), Instance("Zarpie", ("have stripes in their hair", ))),
    (Utterance("Zarpies", "can bounce a ball on their heads"), Instance("Zarpie", ("can bounce a ball on their heads", ))),
    (Utterance("Zarpies", "like to sing"), Instance("Zarpie", ("like to sing", ))),
    (Utterance("Zarpies", "climb tall fences"), Instance("Zarpie", ("climb tall fences", ))),
    (Utterance("Zarpies", "flap their arms when they are happy"), Instance("Zarpie", ("flap their arms when they are happy", ))),
    (Utterance("Zarpies", "have freckles on their feet"), Instance("Zarpie", ("have freckles on their feet", ))),
    (Utterance("Zarpies", "hop over puddles"), Instance("Zarpie", ("hop over puddles", ))),
    (Utterance("Zarpies", "really don't like walking in the mud"), Instance("Zarpie", ("really don't like walking in the mud", ))),
    (Utterance("Zarpies", "draw stars on their knees"), Instance("Zarpie", ("draw stars on their knees", ))),
    (Utterance("Zarpies", "can flip in the air"), Instance("Zarpie", ("can flip in the air", ))),
    (Utterance("Zarpies", "are scared of ladybugs"), Instance("Zarpie", ("are scared of ladybugs", ))),
    (Utterance("Zarpies", "really don't like ice cream"), Instance("Zarpie", ("really don't like ice cream", ))),
    (Utterance("Zarpies", "chase shadows"), Instance("Zarpie", ("chase shadows", ))),
    (Utterance("Zarpies", "babies are wrapped in orange blankets"), Instance("Zarpie", ("babies are wrapped in orange blankets", ))),
    (Utterance("Zarpies", "sleep in tall trees"), Instance("Zarpie", ("sleep in tall trees", )))
)

specific_condition = (
    (Utterance("This Zarpie", "love to eat flowers"), Instance("Zarpie", ("love to eat flowers", ))),
    (Utterance("This Zarpie", "have stripes in their hair"), Instance("Zarpie", ("have stripes in their hair", ))),
    (Utterance("This Zarpie", "can bounce a ball on their heads"), Instance("Zarpie", ("can bounce a ball on their heads", ))),
    (Utterance("This Zarpie", "like to sing"), Instance("Zarpie", ("like to sing", ))),
    (Utterance("This Zarpie", "climb tall fences"), Instance("Zarpie", ("climb tall fences", ))),
    (Utterance("This Zarpie", "flap their arms when they are happy"), Instance("Zarpie", ("flap their arms when they are happy", ))),
    (Utterance("This Zarpie", "have freckles on their feet"), Instance("Zarpie", ("have freckles on their feet", ))),
    (Utterance("This Zarpie", "hop over puddles"), Instance("Zarpie", ("hop over puddles", ))),
    (Utterance("This Zarpie", "really don't like walking in the mud"), Instance("Zarpie", ("really don't like walking in the mud", ))),
    (Utterance("This Zarpie", "draw stars on their knees"), Instance("Zarpie", ("draw stars on their knees", ))),
    (Utterance("This Zarpie", "can flip in the air"), Instance("Zarpie", ("can flip in the air", ))),
    (Utterance("This Zarpie", "are scared of ladybugs"), Instance("Zarpie", ("are scared of ladybugs", ))),
    (Utterance("This Zarpie", "really don't like ice cream"), Instance("Zarpie", ("really don't like ice cream", ))),
    (Utterance("This Zarpie", "chase shadows"), Instance("Zarpie", ("chase shadows", ))),
    (Utterance("This Zarpie", "babies are wrapped in orange blankets"), Instance("Zarpie", ("babies are wrapped in orange blankets", ))),
    (Utterance("This Zarpie", "sleep in tall trees"), Instance("Zarpie", ("sleep in tall trees", )))
)

baseline_condition = ()

# Clean Study 6 data

In [9]:
# get observed data
df_raw = pd.read_csv("../data/full_study6.csv")

# select only relevant columns
df_raw = df_raw[["ID", "condition", 
                 "live_caves", "ride_lions", "farm_potatoes", "play_banjo", 
                 "look_left", "clap_three", "smile_sad", "chug_syrup", 
                 "yell_cats", "go_opera", "dance_fire", "sing_songs",
                 "scream_windows", "smell_garbage", "wash_ponds", "paint_hands"]]

# tidy data
df_tidy = pd.melt(df_raw,
                  id_vars = ["ID", "condition"], # do *not* tidy these
                  var_name = "feature", value_name = "prevalence")

# order condition
df_tidy['condition'] = df_tidy['condition'].astype(pd.CategoricalDtype(categories=["generic", "baseline", "specific"],
                                                                       ordered=True))

# convert prevalence to 0-1
df_tidy["prevalence"] = df_tidy["prevalence"].div(100)

# clip prevalence to .01-.99, because the non-uniform beta distribution does not include endpoints
df_tidy["prevalence"] = df_tidy["prevalence"].replace(1, .99)
df_tidy["prevalence"] = df_tidy["prevalence"].replace(0, .01)

# get all test features
all_test_features = df_tidy["feature"].unique()

# export df_tidy to csv
df_tidy.to_csv("scratch/study 6/df_tidy.csv", index=False)

# Get coherence distributions from each model for each condition

In [ ]:
###################################
## Note: this takes a while to run, so I have stored the output in a pickle file. If you want to re-run, uncomment the code below and run it once. Then comment it out again to avoid re-running each time.
###################################

In [ ]:
# # #### Pragmatic listener coherence distributions

# # fix inverse temperature at something reasonable, to reduce number of parameters to estimate
# inv_temp = 20

# # get fixed coherence dists
# dist_coherence_generic_prag = pragmatic_listener(
#     generic_condition, 
#     inv_temp = inv_temp,
#     only_return_coherence = True)

# dist_coherence_specific_prag = pragmatic_listener(
#     specific_condition, 
#     inv_temp = inv_temp,
#     only_return_coherence = True)

# dist_coherence_baseline_prag = pragmatic_listener(
#     baseline_condition, 
#     inv_temp = inv_temp,
#     only_return_coherence = True)

# # store output so we don't have to re-run pragmatic listener each time
# with open("scratch/study 6/dist_coherence_generic_prag.pkl", "wb") as f:
#     pickle.dump(dist_coherence_generic_prag, f)

# with open("scratch/study 6/dist_coherence_specific_prag.pkl", "wb") as f:
#     pickle.dump(dist_coherence_specific_prag, f)

# with open("scratch/study 6/dist_coherence_baseline_prag.pkl", "wb") as f:
#     pickle.dump(dist_coherence_baseline_prag, f)

In [ ]:
# #### Literal listener coherence distributions

# # get coherence dists from literal listener
# dist_coherence_generic_lit = literal_listener(
#     generic_condition, 
#     only_return_coherence = True)

# dist_coherence_specific_lit = literal_listener(
#     specific_condition, 
#     only_return_coherence = True)

# dist_coherence_baseline_lit = literal_listener(
#     baseline_condition, 
#     only_return_coherence = True)

# # store output so we don't have to re-run literal listener each time
# with open("scratch/study 6/dist_coherence_generic_lit.pkl", "wb") as f:
#     pickle.dump(dist_coherence_generic_lit, f)

# with open("scratch/study 6/dist_coherence_specific_lit.pkl", "wb") as f:
#     pickle.dump(dist_coherence_specific_lit, f)

# with open("scratch/study 6/dist_coherence_baseline_lit.pkl", "wb") as f:
#     pickle.dump(dist_coherence_baseline_lit, f)

In [ ]:
# #### Base model coherence distributions

# # get coherence dists from base model (lesioning meaning)
# dist_coherence_generic_base = literal_listener(
#     generic_condition, 
#     only_return_coherence = True,
#     lesioned_meaning = True)

# dist_coherence_specific_base = literal_listener(
#     specific_condition, 
#     only_return_coherence = True,
#     lesioned_meaning = True)

# dist_coherence_baseline_base = literal_listener(
#     baseline_condition, 
#     only_return_coherence = True,
#     lesioned_meaning = True)


# # store output so we don't have to re-run base model each time
# with open("scratch/study 6/dist_coherence_generic_base.pkl", "wb") as f:
#     pickle.dump(dist_coherence_generic_base, f)

# with open("scratch/study 6/dist_coherence_specific_base.pkl", "wb") as f:
#     pickle.dump(dist_coherence_specific_base, f)

# with open("scratch/study 6/dist_coherence_baseline_base.pkl", "wb") as f:
#     pickle.dump(dist_coherence_baseline_base, f)

In [10]:
#############################
# Load coherence distributions from each model for each condition
#############################

# load pragmatic listener coherence outputs
with open("scratch/study 6/dist_coherence_generic_prag.pkl", "rb") as f: 
    dist_coherence_generic_prag = pickle.load(f)

with open("scratch/study 6/dist_coherence_specific_prag.pkl", "rb") as f:
    dist_coherence_specific_prag = pickle.load(f)

with open("scratch/study 6/dist_coherence_baseline_prag.pkl", "rb") as f:
    dist_coherence_baseline_prag = pickle.load(f)

# load literal listener coherence outputs
with open("scratch/study 6/dist_coherence_generic_lit.pkl", "rb") as f: 
    dist_coherence_generic_lit = pickle.load(f)

with open("scratch/study 6/dist_coherence_specific_lit.pkl", "rb") as f:
    dist_coherence_specific_lit = pickle.load(f)

with open("scratch/study 6/dist_coherence_baseline_lit.pkl", "rb") as f:
    dist_coherence_baseline_lit = pickle.load(f)
    
# load base model coherence outputs
with open("scratch/study 6/dist_coherence_generic_base.pkl", "rb") as f: 
    dist_coherence_generic_base = pickle.load(f)

with open("scratch/study 6/dist_coherence_specific_base.pkl", "rb") as f:
    dist_coherence_specific_base = pickle.load(f)

with open("scratch/study 6/dist_coherence_baseline_base.pkl", "rb") as f:
    dist_coherence_baseline_base = pickle.load(f)

# Fit linking function parameters for each model

In [ ]:
###################################
## Fit parameters in each model for each feature
###################################

In [11]:
##### Define helper functions

# put row-wise loop into separate function to speed up runtime
def observe_response(
    condition, 
    prevalence,
    feature_kind_linked_prevalence, 
    feature_not_kind_linked_prevalence, 
    model,
    simulate_response = False
):
    if model == "pragmatic": 
        # get the coherence distribution for the condition
        if condition == "generic":
            dist_coherence = dist_coherence_generic_prag

        elif condition == "specific":
            dist_coherence = dist_coherence_specific_prag
            
        elif condition == "baseline":
            dist_coherence = dist_coherence_baseline_prag
        else:
            raise ValueError("Condition must be 'generic', 'specific', or 'baseline'")

    elif model == "literal":
        # get the coherence distribution for the condition
        if condition == "generic":
            dist_coherence = dist_coherence_generic_lit

        elif condition == "specific":
            dist_coherence = dist_coherence_specific_lit
            
        elif condition == "baseline":
            dist_coherence = dist_coherence_baseline_lit
        else:
            raise ValueError("Condition must be 'generic', 'specific', or 'baseline'")
            
    elif model == "base":
        # get the coherence distribution for the condition
        if condition == "generic":
            dist_coherence = dist_coherence_generic_base

        elif condition == "specific":
            dist_coherence = dist_coherence_specific_base
            
        elif condition == "baseline":
            dist_coherence = dist_coherence_baseline_base
        else:
            raise ValueError("Condition must be 'generic', 'specific', or 'baseline'")
    else:
        raise ValueError("Model must be 'pragmatic', 'literal', or 'base'")

    # flip on kind-linked status using a sampled coherence from the pragmatic listener
    coherence = dist_coherence.sample() # this might just be the EV of coherence
    feature_is_kind_linked = flip(coherence)
    
    # get response distribution
    if feature_is_kind_linked:
        response_dist = feature_kind_linked_prevalence
    else:
        response_dist = feature_not_kind_linked_prevalence
    
    if simulate_response:
        # simulate a response
        response = response_dist.sample()
        return response
    else:
        # observe actual response (this reweights the distribution)
        response_dist.observe(prevalence)


# estimate model parameters
def param_estimation(data_feature: tuple[dict, ...],
                     **kwargs):
    
    # sample parameters - manually set initial value at 1 (this is a reasonable place to start)
    feature_kind_linked_prevalence_alpha = Uniform(0, 10).fit(initial_value = 1)
    feature_kind_linked_prevalence_beta = Uniform(0, 10).fit(initial_value = 1)
    feature_not_kind_linked_prevalence_alpha = Uniform(0, 10).fit(initial_value = 1)
    feature_not_kind_linked_prevalence_beta = Uniform(0, 10).fit(initial_value = 1)
    
    # construct distributions
    feature_kind_linked_prevalence = Beta(feature_kind_linked_prevalence_alpha,
                                          feature_kind_linked_prevalence_beta)
    feature_not_kind_linked_prevalence = Beta(feature_not_kind_linked_prevalence_alpha,
                                              feature_not_kind_linked_prevalence_beta)
    
    # compare observed responses to expected responses
    for row in data_feature:
        observe_response(
            row["condition"], row["prevalence"],
            feature_kind_linked_prevalence, feature_not_kind_linked_prevalence,
            **kwargs
        )
    
    # return parameters
    return hashabledict(
        feature_kind_linked_prevalence_alpha = feature_kind_linked_prevalence_alpha,
        feature_kind_linked_prevalence_beta = feature_kind_linked_prevalence_beta,
        feature_not_kind_linked_prevalence_alpha = feature_not_kind_linked_prevalence_alpha,
        feature_not_kind_linked_prevalence_beta = feature_not_kind_linked_prevalence_beta
    )

In [12]:
### pull MMAP inference out of for loop to speed up runtime
mmap_inference = MaximumMarginalAPosteriori(
    param_estimation,
    maximum_likelihood=True,
    max_retries=1,
    seed=12938
)

In [ ]:
###################################
# ## Note: this takes a while to run, so I have stored the output in a pickle file. If you want to re-run, uncomment the code below and run it once. Then comment it out again to avoid re-running each time.
###################################

In [13]:
# ### Pragmatic listener

# # initialize df to store results
# mle_per_feature_prag = pd.DataFrame(columns = ["feature"])
# print("fitting pragmatic listener:")

# # for each feature...
# for feature in all_test_features:
    
#     print("- feature == " + feature)
    
#     # filter df to that feature
#     df_feature = df_tidy[df_tidy["feature"]==feature]
#     # convert df to tuple
#     data_feature = tuple(hashabledict(row) for row in df_feature.to_dict(orient="records"))
    
#     # calculate best parameters using MLE, given prevalence responses
#     result = mmap_inference.run(data_feature, model="pragmatic")
    
#     # convert to df, note which feature was used
#     new_row = dist_to_df(result).assign(feature = feature)
#     # add to results df
#     mle_per_feature_prag = pd.concat([mle_per_feature_prag, new_row], ignore_index = True)

# # store output so we don't have to re-run parameter fitting each time
# mle_per_feature_prag.to_csv('scratch/study 6/mle_per_feature_prag.csv', index=False)

# print(mle_per_feature_prag)

fitting pragmatic listener:
- feature == live_caves


KeyboardInterrupt: 

In [ ]:
# ### Literal listener

# # initialize df to store results
# mle_per_feature_lit = pd.DataFrame(columns = ["feature"])
# print("fitting literal listener:")

# # for each feature...
# for feature in all_test_features:
    
#     print("- feature == " + feature)
    
#     # filter df to that feature
#     df_feature = df_tidy[df_tidy["feature"]==feature]
#     # convert df to tuple
#     data_feature = tuple(hashabledict(row) for row in df_feature.to_dict(orient="records"))
    
#     # calculate best parameters using MLE, given prevalence responses
#     result = mmap_inference.run(data_feature, model="literal")
    
#     # convert to df, note which feature was used
#     new_row = dist_to_df(result).assign(feature = feature)
#     # add to results df
#     mle_per_feature_lit = pd.concat([mle_per_feature_lit, new_row], ignore_index = True)

# # store output so we don't have to re-run parameter fitting each time
# mle_per_feature_lit.to_csv('scratch/study 6/mle_per_feature_lit.csv', index=False)

In [ ]:
# ### Base model

# # initialize df to store results
# mle_per_feature_base = pd.DataFrame(columns = ["feature"])

# print("fitting base model:")

# # for each feature...
# for feature in all_test_features:
    
#     print("- feature == " + feature)
    
#     # filter df to that feature
#     df_feature = df_tidy[df_tidy["feature"]==feature]
#     # convert df to tuple
#     data_feature = tuple(hashabledict(row) for row in df_feature.to_dict(orient="records"))
    
#     # calculate best parameters using MLE, given prevalence responses
#     # this is passing in arguments to param_estimation
#     result = mmap_inference.run(data_feature, model="base") 
    
#     # convert to df, note which feature was used
#     new_row = dist_to_df(result).assign(feature = feature)
#     # add to results df
#     mle_per_feature_base = pd.concat([mle_per_feature_base, new_row], ignore_index = True)

# # store output so we don't have to re-run parameter fitting each time
# mle_per_feature_base.to_csv('scratch/study 6/mle_per_feature_base.csv', index=False)

In [14]:
########
# Load MLE results for each feature and condition from each model (from linking function)
########
mle_per_feature_prag = pd.read_csv('scratch/study 6/mle_per_feature_prag.csv')
mle_per_feature_lit = pd.read_csv('scratch/study 6/mle_per_feature_lit.csv')
mle_per_feature_base = pd.read_csv('scratch/study 6/mle_per_feature_base.csv')